# 01 - Data Cleaning
Pipeline: load raw tables -> inspect & clean each table -> join item-level & order-level -> save to `data/processed/`.

## Setup

In [1]:
from src.data_loader import *
from src.data_cleaning import *

In [2]:
data_dir = 'data/raw'
tables = load_raw_tables(data_dir)

Loaded customers: 99,441 lines, 5 columns
Loaded geolocation: 1,000,163 lines, 5 columns
Loaded order_items: 112,650 lines, 7 columns
Loaded order_payments: 103,886 lines, 5 columns
Loaded order_reviews: 99,224 lines, 7 columns
Loaded orders: 99,441 lines, 8 columns
Loaded products: 32,951 lines, 9 columns
Loaded sellers: 3,095 lines, 4 columns
Loaded category_translation: 71 lines, 2 columns


## 1. Orders

In [3]:
orders = tables['orders']
orders.isna().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

In [4]:
violations_report = check_timestamp_violations(orders)
print(violations_report)

                                                rule  n_checked  n_violations  \
0      order_purchase_timestamp <= order_approved_at      99281             0   
1  order_approved_at <= order_delivered_carrier_date      97644          1359   
2  order_delivered_carrier_date <= order_delivere...      96475            23   
3  order_purchase_timestamp <= order_delivered_ca...      97658           166   
4  order_purchase_timestamp <= order_delivered_cu...      96476             0   

   pct_violations  
0           0.000  
1           1.392  
2           0.024  
3           0.170  
4           0.000  


In [5]:
orders = timestamp_invalid_flag(orders)
print(f"Total flagged orders: {orders['is_invalid_timestamps'].sum()}")

Total flagged orders: 1382


In [6]:
# Sync cleaned table back immediately to avoid stale references later
tables['orders'] = orders

## 2. Customers

In [7]:
customers = tables['customers']
customers.isna().sum()

customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

In [8]:
customers = clean_customers(customers)

In [9]:
print(f"Total row nums: {len(customers)}")
print(f"customer_id unique ratio: {customers['customer_id'].nunique() / len(customers):.4f}")
print(f"customer_unique_id unique ratio: {customers['customer_unique_id'].nunique() / len(customers):.4f}")

Total row nums: 99441
customer_id unique ratio: 1.0000
customer_unique_id unique ratio: 0.9664


In [10]:
tables['customers'] = customers

## 3. Geolocation

In [11]:
geolocation = tables['geolocation']
geolocation.isna().sum()

geolocation_zip_code_prefix    0
geolocation_lat                0
geolocation_lng                0
geolocation_city               0
geolocation_state              0
dtype: int64

In [12]:
geolocation = clean_geolocation(geolocation)
geolocation.head()

Remove 42 coordinates outside of Brazil
Aggregate 1,000,121 lines into 19,010 distinct zip code


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,01001,-23.550190,-46.634024,Sao Paulo,SP
1,01002,-23.548146,-46.634979,Sao Paulo,SP
2,01003,-23.548994,-46.635731,Sao Paulo,SP
3,01004,-23.549799,-46.634757,Sao Paulo,SP
4,01005,-23.549456,-46.636733,Sao Paulo,SP


In [13]:
tables['geolocation'] = geolocation

## 4. Products

In [14]:
products = tables['products']
category_name_translation = tables['category_translation']
products.isna().sum()

product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

In [15]:
products.head()

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0


In [16]:
products[products['product_weight_g'].isna()]

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
8578,09ff539a621711667c43eba6a3bd8466,bebes,60.0,865.0,3.0,NaN,NaN,NaN,NaN
18851,5eb564652db742ff8f28759cd8d2652a,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [17]:
products = clean_products(products, category_name_translation)
products.isna().sum()

Filled missing value at product_weight_g by median value
Filled missing value at product_length_cm by median value
Filled missing value at product_height_cm by median value
Filled missing value at product_width_cm by median value


product_id                         0
product_category_name              0
product_name_length              610
product_description_length       610
product_photos_qty               610
product_weight_g                   0
product_length_cm                  0
product_height_cm                  0
product_width_cm                   0
product_category_name_english      0
dtype: int64

In [18]:
products['product_category_name_english'].value_counts().head(10)

product_category_name_english
bed_bath_table           3029
sports_leisure           2867
furniture_decor          2657
health_beauty            2444
housewares               2335
auto                     1900
computers_accessories    1639
toys                     1411
watches_gifts            1329
telephony                1134
Name: count, dtype: int64

In [19]:
tables['products'] = products

## 5. Order Items

In [20]:
order_items = tables['order_items']
order_items.head(5)

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [21]:
order_items[order_items['price'] <= 0]

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value


In [22]:
order_items[order_items['freight_value'] < 0]

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value


No invalid price/freight values, no duplicates -> only need to flag price outliers (IQR).

In [23]:
order_items = flag_order_items_outlier(order_items)

Flagged 8427 lines (7.48%) has price > 277.4 (IQR)


In [24]:
tables['order_items'] = order_items

## 6. Order Payments

In [25]:
order_payments = tables['order_payments']
order_payments.isna().sum()

order_id                0
payment_sequential      0
payment_type            0
payment_installments    0
payment_value           0
dtype: int64

In [26]:
order_payments.head()

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


In [27]:
order_payments[order_payments['payment_installments'] == 0]
# payment_installments = 0 but payment_type is not voucher => data entry error

,order_id,payment_sequential,payment_type,payment_installments,payment_value
46982,744bade1fcf9ff3f31d860ace076d422,2,credit_card,0,58.69
79014,1a57108394169c0b47d8f876acc9ba2d,2,credit_card,0,129.94


In [28]:
order_payments[order_payments['payment_value'] < 0]
# No negative payment value

,order_id,payment_sequential,payment_type,payment_installments,payment_value


In [29]:
order_payments['payment_type'].unique()

<ArrowStringArray>
['credit_card', 'boleto', 'voucher', 'debit_card', 'not_defined']
Length: 5, dtype: str

In [30]:
order_payments = clean_order_payments(order_payments)

In [31]:
order_payments['payment_type'].unique()

<ArrowStringArray>
['credit_card', 'boleto', 'voucher', 'debit_card', 'other']
Length: 5, dtype: str

In [32]:
order_payments[order_payments['payment_installments'] == 0]

,order_id,payment_sequential,payment_type,payment_installments,payment_value


Note: `order_payments` is kept separate from `tables` (not synced back) because `join_order_level()` takes it as its own parameter — it is aggregated at the order level rather than merged at item level.

## 7. Order Reviews

In [33]:
order_reviews = tables['order_reviews']
order_reviews.isna().sum()

review_id                      0
order_id                       0
review_score                   0
review_comment_title       87656
review_comment_message     58247
review_creation_date           0
review_answer_timestamp        0
dtype: int64

In [34]:
order_reviews[~order_reviews['review_score'].between(1, 5)]

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp


All reviews' scores are between 1-5 => valid.

In [35]:
order_reviews[order_reviews['review_creation_date'] > order_reviews['review_answer_timestamp']]

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp


No illogical timestamps (review_creation_date later than review_answer_timestamp).

In [36]:
order_reviews = clean_order_reviews(order_reviews)

Removed 551 reviews with same order id (keep the lastest), -Keeping 98673 unchanged lines


In [37]:
print(f"Percentage of reviews with a comment: {order_reviews['has_comment'].mean() * 100:.1f}%")

Percentage of reviews with a comment: 41.3%


In [38]:
tables['order_reviews'] = order_reviews

## 8. Sellers

In [39]:
sellers = tables['sellers']
print(sellers.isna().sum())
print('Duplicated sellers:', sellers['seller_id'].duplicated().sum())

seller_id                 0
seller_zip_code_prefix    0
seller_city               0
seller_state              0
dtype: int64
Duplicated sellers: 0


No NA values, no duplicated sellers.

In [40]:
sellers = clean_sellers(sellers)

In [41]:
tables['sellers'] = sellers

## 9. Sanity Check Before Join
Confirm every table in `tables` was actually replaced by its cleaned version (catches stale/None references before they cause a cryptic merge error).

In [42]:
required = ["order_items", "orders", "customers", "products", "sellers", "order_reviews"]
for key in required:
    assert tables.get(key) is not None, f"tables['{key}'] is None — did you forget to sync it back?"
print("All required tables are present and non-null.")

All required tables are present and non-null.


## 10. Join Item-Level & Order-Level

In [43]:
item_level = join_item_level(tables)
order_level = join_order_level(item_level, order_payments)

join_item_level: 112,650 line (item-level), 98,666 unique orders
join_order_level: 98,666 orders (order-level)


## 11. Save to `data/processed/`

In [44]:
item_level.to_parquet("data/processed/item_level.parquet", index=False)
order_level.to_parquet("data/processed/order_level.parquet", index=False)